# Projekt 03 (final): Einkommensvorhersage auf echten Zensusdaten

**Ziel:** Ein vollstaendiges, ehrliches ML-Projekt auf einem echten, "unordentlichen"
Datensatz — gemischte Feature-Typen, fehlende Werte, Klassenungleichgewicht — inklusive
der zwei Themen, die in der Praxis oft fehlen: **Schwellenwahl nach Kosten** und ein
**Fairness-Check**.

**Daten:** UCI **Adult / Census Income** (48.842 Personen, US-Zensus 1994), ueber
`sklearn.datasets.fetch_openml("adult", version=2)` — kein manueller Download noetig,
sklearn cached die Daten lokal (nicht im Repo!). Zielvariable: Einkommen `>50K` vs. `<=50K`
(~24 % Positive — moderat unausgewogen). Enthaelt echte fehlende Werte (`workclass`,
`occupation`, `native-country`) und eine Mischung aus numerischen und kategorialen Spalten
inkl. sensibler Attribute (`sex`, `race`) — das macht den Datensatz zum Klassiker fuer
Fairness-Diskussionen in der ML-Praxis.

Vorwissen: das ganze Modul-Skript, Projekt 02 (Pipelines, CV, Tuning), Skript 2.6
(unausgewogene Klassen) und 3.1 (Interpretation).

## 1. Daten laden & explorieren

**Aufgabe:** Lade den Datensatz, wirf einen Blick auf Form, Spaltentypen, fehlende Werte
und die Zielverteilung.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 2. Aufbereitung: Zielvariable & Features

**Aufgabe:**
1. Erzeuge `y` als 0/1: `1` fuer `">50K"`, `0` fuer `"<=50K"`.
2. Entferne `fnlwgt` (eine Zensus-Gewichtungsspalte, kein inhaltliches Merkmal) und
   `education` (redundant zu `education-num`, das dieselbe Information bereits
   ordinal als Zahl traegt) aus den Features.
3. Definiere zwei Listen: `numerische_features` und `kategoriale_features`
   (alle uebrigen Spalten, sinnvoll nach Dtype sortiert).

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 3. Train/Test-Split

**Aufgabe:** Stratifizierter Split, `test_size=0.2`, `random_state=42`. Testsatz bis
Schritt 6 nicht anfassen.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 4. Preprocessing-Pipeline

**Aufgabe:** Baue mit `ColumnTransformer` (Skript 2.4) eine Preprocessing-Pipeline:

- numerische Spalten: `SimpleImputer(strategy="median")` + `StandardScaler()`
  (hier zwar keine fehlenden numerischen Werte, aber robuste Gewohnheit)
- kategoriale Spalten: `SimpleImputer(strategy="most_frequent")` + `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`
  (fehlende Werte wie `workclass` werden so durch den haeufigsten Wert ersetzt;
  `sparse_output=False`, weil `HistGradientBoostingClassifier` in Schritt 6 dichte statt duennbesetzte Matrizen braucht)

Kombiniere beide Zweige zu `vorverarbeitung = ColumnTransformer([...])`.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 5. Baseline

**Aufgabe:** Trainiere einen `DummyClassifier(strategy="most_frequent")` als triviale Vergleichsbasis und gib seine Test-Accuracy aus (er sagt einfach immer `<=50K`).


In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 6. Modellvergleich unter Klassenungleichgewicht

Bei ~24 % Positiven ist **Accuracy irrefuehrend** (Skript 1.5/2.6) — wir vergleichen
per **PR-AUC** (`scoring="average_precision"`), die bei unausgewogenen Klassen
aussagekraeftiger ist als ROC-AUC.

**Aufgabe:** Baue drei Pipelines (`vorverarbeitung` + Klassifikator), jeweils mit
`class_weight="balanced"` (Skript 2.6):

- `LogisticRegression(max_iter=1000, class_weight="balanced")`
- `RandomForestClassifier(random_state=42, class_weight="balanced")`
- `HistGradientBoostingClassifier(random_state=42, class_weight="balanced")`

Vergleiche mit `StratifiedKFold(5)` + `cross_val_score(scoring="average_precision")`
auf den Trainingsdaten, stelle die Ergebnisse als Boxplot dar.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 7. Tuning des besten Modells

**Aufgabe:** Waehle das beste Modell aus Schritt 6 und tune es mit `GridSearchCV`
(`cv=cv`, `scoring="average_precision"`) auf den Trainingsdaten. Ein sinnvolles Grid
fuer `HistGradientBoostingClassifier` waere z. B. `clf__max_iter`, `clf__learning_rate`,
`clf__max_leaf_nodes` — passe an dein gewaehltes Modell an.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 8. Die einmalige Testauswertung

**Aufgabe:** `bestes_modell = suche.best_estimator_`, Vorhersagen auf `X_test` bei
Standard-Schwelle 0,5, Classification Report, PR-Kurve + PR-AUC auf dem Testsatz.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## 9. Schwellenwahl per Kostenrechnung

Die 0,5-Schwelle ist Konvention, keine Notwendigkeit (Skript 2.6/3.2). Szenario: Ein
Programm fuer Finanzberatungs-Outreach will Personen mit `>50K` Einkommen ansprechen.
Ein **Fehlalarm** (FP: Kontakt, aber eigentlich `<=50K`) kostet **50 $** (verschwendeter
Aufwand). Ein **uebersehener Fall** (FN: `>50K`, aber nicht kontaktiert) kostet **200 $**
(entgangenes Geschaeft) — vier mal so teuer wie ein Fehlalarm.

**Aufgabe:** Berechne fuer ein Raster von Schwellen (`np.linspace(0.01, 0.99, 99)`) die
Anzahl FP und FN auf dem Testsatz (aus `y_proba`) und damit die Gesamtkosten
`50 * FP + 200 * FN`. Finde die kostenminimale Schwelle und vergleiche sie mit 0,5.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


**Aufgabe:** Warum liegt die kostenminimale Schwelle unterhalb von 0,5, obwohl das
Programm noch dazu ohnehin schon per `class_weight="balanced"` trainiert wurde? (Hinweis:
Skript 2.6 — Klassengewichte und Schwellenwahl loesen unterschiedliche Probleme.)

*(Deine Notiz hier ...)*

## 10. Fairness-Check: Fehlerraten nach Geschlecht

Der Datensatz enthaelt `sex` als Feature — das Modell hat also direkten oder indirekten
(ueber korrelierte Features) Zugriff darauf. **Aufgabe:** Berechne bei der
kostenminimalen Schwelle aus Schritt 9 fuer beide Gruppen (`Male`, `Female`) getrennt
Recall und False-Positive-Rate auf dem Testsatz und stelle sie nebeneinander dar.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


**Aufgabe:** Beschreibe den Unterschied zwischen den Gruppen in 2-3 Saetzen. Ist er
durch die unterschiedliche Basisrate (`>50K`-Anteil) erklaerbar, oder bleibt eine
Kluft, die auf unterschiedliche Fehlerbehandlung hindeutet? Was wuerdest du einem
Team empfehlen, das dieses Modell operativ einsetzen will?

*(Deine Notiz hier ...)*

## 11. Permutation Importance

**Aufgabe:** Berechne `permutation_importance` fuer `bestes_modell` **direkt auf den
rohen Testfeatures** `X_test` (die Pipeline uebernimmt die Vorverarbeitung intern —
so bekommst du Wichtigkeiten pro **Originalspalte**, nicht pro One-Hot-Dummy).
Plotte alle Features als horizontales Balkendiagramm.

In [ ]:
# Dein Code hier. (Musterloesung: loesung/loesung.ipynb)


## Geschafft — was du jetzt kannst

- Ein komplettes, realistisches ML-Projekt end-to-end durchziehen: gemischte Typen,
  fehlende Werte, Klassenungleichgewicht
- `ColumnTransformer` fuer heterogenes Preprocessing in einer Pipeline einsetzen
- Modelle unter Ungleichgewicht fair vergleichen (PR-AUC statt Accuracy)
- Eine Entscheidungsschwelle **nach Kosten**, nicht nach Konvention waehlen
- Ein Modell auf **Fairness zwischen Untergruppen** pruefen, bevor es eingesetzt wird
- Wichtigkeiten auf Originalfeature-Ebene interpretieren

**Reflexionsfragen:**

1. Warum reicht `class_weight="balanced"` allein nicht, um faire Fehlerraten zwischen
   den Geschlechtergruppen zu garantieren?
2. Was wuerde passieren, wenn du `sex` als Feature komplett entfernst — wuerde das die
   in Schritt 10 gefundene Kluft zuverlaessig beseitigen? (Stichwort: korrelierte
   Proxy-Features wie `relationship` oder `occupation`.)
3. Der Datensatz stammt von 1994. Welche Gefahr birgt es, ein auf solchen Daten
   trainiertes Modell heute operativ einzusetzen?

**Bonusaufgaben:**
1. Trainiere `HistGradientBoostingClassifier` mit `categorical_features="from_dtype"`
   *ohne* One-Hot-Encoding (direkt auf den pandas-`category`-Spalten) — vergleiche
   PR-AUC und Trainingszeit mit der One-Hot-Variante.
2. Wiederhole den Fairness-Check fuer `race` statt `sex`.
3. Probiere `CalibratedClassifierCV` und pruefe, ob die kostenminimale Schwelle aus
   Schritt 9 sich fuer das kalibrierte Modell verschiebt.